In [7]:

import torch
import pet
import numpy as np
from src.pet import PET, PETUtilityWrapper, PETMLIPWrapper
from src.hypers import load_hypers_from_file

state_path = 'test_pet/results/test/best_val_rmse_both_model_state_dict'
state = torch.load(state_path, map_location='cpu')
print(state.keys())


HYPERS_PATH = "test_pet/results/test/hypers_used.yaml"
hypers = load_hypers_from_file(HYPERS_PATH)
ARCHITECTURAL_HYPERS = hypers.ARCHITECTURAL_HYPERS
ALL_SPECIES_PATH = "test_pet/results/test/all_species.npy"
all_species = np.load(ALL_SPECIES_PATH)
device = 'cpu'
FITTING_SCHEME = hypers.FITTING_SCHEME
PATH_TO_MODEL_STATE_DICT = state_path
dtype = torch.float32

model = PET(ARCHITECTURAL_HYPERS, 0.0, len(all_species)).to(device)
model = PETUtilityWrapper(model, FITTING_SCHEME.GLOBAL_AUG)

if hypers.UTILITY_FLAGS.CALCULATION_TYPE == "mlip":
    model = PETMLIPWrapper(
        model, hypers.MLIP_SETTINGS.USE_ENERGIES, hypers.MLIP_SETTINGS.USE_FORCES
    )

model.load_state_dict(torch.load(PATH_TO_MODEL_STATE_DICT))
model = model.to(dtype=dtype)
model.eval()

print(model)


odict_keys(['model.pet_model.embedding.weight', 'model.pet_model.gnn_layers.0.trans_layer.attention.input_linear.weight', 'model.pet_model.gnn_layers.0.trans_layer.attention.input_linear.bias', 'model.pet_model.gnn_layers.0.trans_layer.attention.output_linear.weight', 'model.pet_model.gnn_layers.0.trans_layer.attention.output_linear.bias', 'model.pet_model.gnn_layers.0.trans_layer.norm_attention.weight', 'model.pet_model.gnn_layers.0.trans_layer.norm_attention.bias', 'model.pet_model.gnn_layers.0.trans_layer.norm_mlp.weight', 'model.pet_model.gnn_layers.0.trans_layer.norm_mlp.bias', 'model.pet_model.gnn_layers.0.trans_layer.mlp.0.weight', 'model.pet_model.gnn_layers.0.trans_layer.mlp.0.bias', 'model.pet_model.gnn_layers.0.trans_layer.mlp.3.weight', 'model.pet_model.gnn_layers.0.trans_layer.mlp.3.bias', 'model.pet_model.gnn_layers.0.trans.layers.0.attention.input_linear.weight', 'model.pet_model.gnn_layers.0.trans.layers.0.attention.input_linear.bias', 'model.pet_model.gnn_layers.0.tran

In [8]:
#data loading

from torch_geometric.loader import DataLoader, DataListLoader
from src.data_preparation import get_pyg_graphs, get_compositional_features
from ase.io import read

structures_path = './test_pet/r_water_test.xyz'
structures = read(structures_path, index=":")

graphs = get_pyg_graphs(
    structures,
    all_species,
    ARCHITECTURAL_HYPERS.R_CUT,
    ARCHITECTURAL_HYPERS.USE_ADDITIONAL_SCALAR_ATTRIBUTES,
    ARCHITECTURAL_HYPERS.USE_LONG_RANGE,
    ARCHITECTURAL_HYPERS.K_CUT,
    ARCHITECTURAL_HYPERS.N_TARGETS > 1,
    ARCHITECTURAL_HYPERS.TARGET_INDEX_KEY
)

loader = DataLoader(graphs, batch_size=1, shuffle=False)

for batch in loader:
    t_batch = batch
    break
print(t_batch)

100%|██████████| 1/1 [00:00<00:00, 108.37it/s]

DataBatch(x=[192, 44, 3], central_species=[192], neighbor_species=[192, 44], neighbors_pos=[192, 44], neighbors_index=[44, 192], nums=[192], mask=[192, 44], n_atoms=[1], batch=[192], ptr=[2])


In [9]:
#inference
USE_AUGMENTATION= True
prediction = model(t_batch, augmentation=USE_AUGMENTATION, create_graph=False)
# print(prediction)
energy = prediction[0]
forces = prediction[1]
print("Energy:", energy)
print("Forces:", forces.shape)

torch.Size([192, 1]) torch.Size([192, 1])
torch.Size([192, 128])
Energy: tensor([-331.9635], grad_fn=<SelectBackward0>)
Forces: torch.Size([192, 3])
